# 03 - Explore the raw data
**What it does:** <in your own words, one or two sentences>
**Needs:** notebook 02 has run (tennis_raw.wta_matches exists)
**Produces:** nothing: read-only exploration. Its findings justify the choices in notebooks 04 and 05
**Run after:** 02_load_tennis_bigquery (not part of the rebuild chain; it only reads)

## Install the pinned libraries

In [ ]:
# The repo is public, so requirements.txt can be read straight from GitHub.
REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
TABLE = f"{PROJECT_ID}.tennis_raw.wta_matches"

## What does a row look like?

In [ ]:
run(f"SELECT * FROM `{TABLE}` LIMIT 3").T

## Row count and duplicates
**Finding:** 43,924 rows, equal to the sum of the yearly files in the download manifest, and equal to the number of unique (tourney_id, match_num) pairs. No duplicates, and nothing lost between the bucket and BigQuery.

In [ ]:
run(f"""
SELECT COUNT(*) AS rows_total,
       COUNT(DISTINCT CONCAT(tourney_id, '|', match_num)) AS unique_matches
FROM `{TABLE}`
""")

## How much of each column is empty?
**Findings:**
- Serve statistics (aces, double faults, serve points...) are about 11% empty overall.
- `minutes` is about 41% empty.
- Seed and entry columns are mostly empty because most players are unseeded or entered directly, so empty means "none" and not "missing".
- Height is 10.5% (winner) and 14.9% (loser) empty. It does not change, so it can be filled from the same player's other matches.
- Rank and ranking points are 1.4% to 2.8% empty (probably unranked players).
- `score` is never empty, so games and sets can be derived from it.

In [ ]:
cols = [f.name for f in bq.get_table(TABLE).schema]
exprs = ",\n  ".join(
    f"ROUND(100 * COUNTIF(`{c}` IS NULL OR `{c}` = '') / COUNT(*), 1) AS `{c}`" for c in cols
)
missing = run(f"SELECT {exprs} FROM `{TABLE}`").T
missing.columns = ["percent_empty"]
missing.sort_values("percent_empty", ascending=False)

## Is there a real match date?
**Finding:** no. Every tournament has exactly one `tourney_date`, the start of the tournament (usually a Monday, according to the data dictionary). Match days have to be estimated from the round (notebook 05).

In [ ]:
run(f"""
SELECT COUNT(DISTINCT tourney_id) AS tournaments,
       COUNTIF(n_dates > 1) AS tournaments_with_more_than_one_date
FROM (
  SELECT tourney_id, COUNT(DISTINCT tourney_date) AS n_dates
  FROM `{TABLE}` GROUP BY tourney_id
)
""")

## Tournament levels, surfaces and names
**Findings:**
- 2,529 tournament IDs but 1,633 distinct names.
- Level D is the team competition (Fed Cup / Billie Jean King Cup), where each tie has its own name, such as "Fed Cup WG R1: AUS vs ITA". Team format at national venues, so it is excluded.
- Surface has inconsistent case (`Clay` and `clay`), and 115 team-event rows have no surface.

In [ ]:
print(run(f"""
SELECT tourney_level, surface, COUNT(*) AS matches
FROM `{TABLE}` GROUP BY 1, 2 ORDER BY matches DESC
"""))

print(run(f"""
SELECT COUNT(DISTINCT tourney_name) AS distinct_names,
       COUNT(DISTINCT tourney_id) AS distinct_tourney_ids
FROM `{TABLE}`
"""))

## What are the small levels?
**Findings (identified from the tournament names; the data dictionary does not define them):**
- `O` = Olympics; `W` = WTA 125-type events (plus one mislabelled 2015 "WTA Finals"); `35+H` and `50+H` = lower-tier ITF events.
- All are small, mostly one-off editions, so they are excluded.

In [ ]:
run(f"""
SELECT tourney_level, tourney_name, COUNT(*) AS matches,
       MIN(tourney_date) AS first_date, MAX(tourney_date) AS last_date
FROM `{TABLE}`
WHERE tourney_level IN ('W','O','50+H','35+H')
GROUP BY 1, 2 ORDER BY tourney_level, matches DESC
""")

## How well filled are the statistics, by level and by year?
**Findings:**
- `minutes` is empty for essentially every match from 2010 to 2015 and filled from 2016 (0 to 7% empty). It is a 2016-onward feature only.
- Serve statistics are 10 to 13% empty in 2010 to 2014, 22% in 2015, and 0 to 2% from 2016.
- 2026 is about 13% empty (the latest matches are probably not filled in yet).
- Level D has 62% of serve statistics missing.
- The gaps follow time, not chance. Filling them and training across the break could let a model learn the year from which columns exist, which is a form of leakage, so they are not imputed across it.

In [ ]:
print(run(f"""
SELECT tourney_level, COUNT(*) AS matches,
       ROUND(100*COUNTIF(w_svpt IS NULL)/COUNT(*),1) AS pct_no_serve_stats,
       ROUND(100*COUNTIF(minutes IS NULL)/COUNT(*),1) AS pct_no_minutes
FROM `{TABLE}` GROUP BY tourney_level ORDER BY matches DESC
"""))

print(run(f"""
SELECT SUBSTR(tourney_date,1,4) AS year, COUNT(*) AS matches,
       ROUND(100*COUNTIF(w_svpt IS NULL)/COUNT(*),1) AS pct_no_serve_stats,
       ROUND(100*COUNTIF(minutes IS NULL)/COUNT(*),1) AS pct_no_minutes
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY year ORDER BY year
"""))

## Can the tournament ID identify a venue?
**Finding:** no. The ID suffix is stable for some tournaments, but some suffixes cover different cities (for example Tokyo and Osaka share one) and some tournaments change suffix. Names are more stable, although spelled inconsistently ("US Open" and "Us Open"). The venue table is therefore keyed on the cleaned tournament name (lower-cased and trimmed).

In [ ]:
print(run(f"""
SELECT tourney_level,
       COUNT(DISTINCT tourney_id) AS tourney_ids,
       COUNT(DISTINCT SUBSTR(tourney_id, 6)) AS distinct_id_suffixes,
       COUNT(DISTINCT tourney_name) AS distinct_names
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
GROUP BY tourney_level ORDER BY tourney_ids DESC
"""))

run(f"""
SELECT SUBSTR(tourney_id, 6) AS id_suffix,
       COUNT(DISTINCT SUBSTR(tourney_id, 1, 4)) AS years,
       ARRAY_AGG(DISTINCT tourney_name LIMIT 4) AS example_names
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY id_suffix ORDER BY years DESC LIMIT 15
""")

## What kinds of score are there?
**Finding (levels G, PM, P, I, before exclusions):** 37,542 normal scores, 1,173 retirements (`RET`), 261 walkovers (`W/O`) and 22 defaults (`DEF`). A walkover means no match was played.

In [ ]:
run(f"""
SELECT COALESCE(REGEXP_EXTRACT(score, r'([A-Za-z]+(?:/[A-Za-z]+)?)'), '(normal score)') AS tag,
       COUNT(*) AS matches
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
GROUP BY tag ORDER BY matches DESC
""")

## Which events have a round-robin stage?
**Finding:** the United Cup (4 editions, 96 matches, a team event) and one Zhuhai edition (12 matches). In a round robin the round says nothing about the day, so match days cannot be estimated. Rule used in notebook 04: drop every event that has a round-robin stage.

In [ ]:
run(f"""
SELECT LOWER(TRIM(tourney_name)) AS name_key, tourney_level, COUNT(DISTINCT tourney_id) AS editions, COUNT(*) AS matches
FROM `{TABLE}`
WHERE round = 'RR' AND tourney_level IN ('G','PM','P','I')
GROUP BY 1, 2 ORDER BY 1
""")

## Rounds and draw sizes
**Finding:** events come in formats that the first round reveals. First round R128 means a two-week event, R64 a longer one-week event (a 56 or 64 draw with byes), R32 an ordinary one-week event. Grand Slams have 128 draws (64 first-round matches each). This is what notebook 05 uses to estimate match days.

In [ ]:
print(run(f"""
SELECT tourney_level, round, COUNT(*) AS matches, COUNT(DISTINCT tourney_id) AS tournaments
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY 1, 2 ORDER BY 1, matches DESC
"""))

print(run(f"""
SELECT tourney_level, draw_size, COUNT(DISTINCT tourney_id) AS tournaments
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I')
GROUP BY 1, 2 ORDER BY 1, 2
"""))

## Values in the seed columns that are not numbers
**Finding:** 97 values (winner: Q 15, WC 10; loser: Q 44, WC 20, LL 8) are entry codes (qualifier, wild card, lucky loser) sitting in the seed column. None of those rows had an entry value already. Notebook 04 moves them to the entry column.

In [ ]:
run(f"""
SELECT 'winner' AS side, TRIM(winner_seed) AS code, NULLIF(TRIM(winner_entry), '') AS existing_entry, COUNT(*) AS n
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
  AND NULLIF(TRIM(winner_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(winner_seed) AS INT64) IS NULL
GROUP BY 2, 3
UNION ALL
SELECT 'loser', TRIM(loser_seed), NULLIF(TRIM(loser_entry), ''), COUNT(*)
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I')
  AND NULLIF(TRIM(loser_seed), '') IS NOT NULL AND SAFE_CAST(TRIM(loser_seed) AS INT64) IS NULL
GROUP BY 2, 3
ORDER BY n DESC
""")

## Tournaments whose start date falls in a different year from their season
The year in `tourney_id` is the season the source assigns. A tournament that starts in the last days of December can belong to the next season, so its `tourney_date` year differs from its `tourney_id` year (and from the year of the file it is stored in).

**Finding:** <fill in after running: which tournaments, how many matches, and which start dates>

In [ ]:
run(f"""
SELECT SUBSTR(tourney_id, 1, 4) AS id_year, SUBSTR(tourney_date, 1, 4) AS date_year,
       tourney_name, tourney_date, COUNT(*) AS matches
FROM `{TABLE}`
WHERE SUBSTR(tourney_id, 1, 4) <> SUBSTR(tourney_date, 1, 4)
GROUP BY 1, 2, 3, 4 ORDER BY tourney_date
""")

## Notes from the data dictionary
- `tourney_date` is the start of the tournament week. `tourney_id` has a predictable year and nothing else.
- Draw size is often rounded up to the next power of 2.
- Rank, rank points and age are as of the tournament start, so they are known before the match.
- Seed and entry are known before the match. Entry codes: WC wild card, Q qualifier, LL lucky loser, PR protected ranking.
- `match_num` is not reliable for ordering matches.
- Level codes: G Slams, PM Premier Mandatory, P Premier, I International, F season-ending events, D team events.

## Leakage warnings that shape the design
- The table has a winner side and a loser side. Building features from those columns would build the label into the layout, so each match is reshaped into player A versus player B (notebook 09).
- The score, minutes and all `w_` and `l_` statistics are outcomes of the match. They can only be used as history for later matches, never as features of the same match.

## Scope decision
Modelling scope: levels G, PM, P, I, excluding every event with a round-robin stage (38,860 matches, 872 tournament editions). Excluded: D (team format), O (Olympics), W and the ITF levels (few, one-off editions), F (season-ending round robin, host city changes yearly).